In [ ]:
import re, pathlib, pymupdf, docx

def load_docs(folder):
    docs = []
    for path in sorted(pathlib.Path(folder).glob("*")):
        if path.suffix == ".pdf":
            for i, page in enumerate(pymupdf.open(path)):
                docs.append({"source": path.name, "page": i + 1, "text": page.get_text()})
        elif path.suffix == ".docx":
            text = "\n".join(p.text for p in docx.Document(path).paragraphs)
            docs.append({"source": path.name, "page": None, "text": text})
    return docs

def clean(text):
    text = re.sub(r"\.{3,}", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def is_front_matter(raw_text):
    return len(re.findall(r"\.{5,}", raw_text)) >= 5

def is_useful(text, min_words=25, min_letters=0.35):
    words = text.split()
    if len(words) < min_words:
        return False
    letter_words = re.findall(r"[A-Za-z]{3,}", text)
    return len(letter_words) / len(words) >= min_letters

def chunk_text(text, size=250, overlap=40):
    words = text.split()
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, max(len(words) - overlap, 1), step)]

def make_chunks(docs, size=250, overlap=40):
    chunks, dropped = [], []
    for d in docs:
        if is_front_matter(d["text"]):
            dropped.append({"source": d["source"], "page": d["page"], "text": d["text"][:80]})
            continue
        for c in chunk_text(clean(d["text"]), size, overlap):
            item = {"source": d["source"], "page": d["page"], "text": c}
            (chunks if is_useful(c) else dropped).append(item)
    print(f"kept {len(chunks)}, dropped {len(dropped)}")
    return chunks, dropped

In [ ]:
docs = load_docs("papers")
chunks, dropped = make_chunks(docs)
chunks[5]

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-small-en-v1.5"
model = SentenceTransformer(MODEL_NAME)

texts = [c["text"] for c in chunks]
embeddings = model.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=True)
embeddings = np.asarray(embeddings, dtype="float32")
embeddings.shape

In [ ]:
import faiss

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
index.ntotal

In [ ]:
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def search(query, k=5):
    q = model.encode([QUERY_PREFIX + query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    for rank, (s, i) in enumerate(zip(scores[0], ids[0]), 1):
        c = chunks[i]
        print(f"\n#{rank} [{s:.3f}] {c['source']} p.{c['page']}")
        print(c["text"][:400], "...")

search("Which statistical test compares the conditions?")